<a href="https://colab.research.google.com/github/linabash/Selected-Topics-YOLO11-s0r1-original-data/blob/main/Copy_of_Fire_Smoke_orgData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training notebook: one model, 30 epochs (Colab T4)

This notebook trains **one model** on all the training images (8,826 images), evaluates it on the test split, and saves everything to your own Google Drive.

## How to run it

1. From the File menu choose **Save a copy in Drive**, so you work on your own copy.
2. Change only two lines in the first code cell: `MODEL_TO_RUN` and `SEED`.
3. From the Runtime menu choose **Change runtime type**, then **T4 GPU**.
4. From the Runtime menu choose **Run all**, and click **Allow** when it asks for Google Drive access.
5. Keep the browser tab open until it finishes.
6. If Colab disconnects, open the notebook and click **Run all** again. It resumes from the last saved epoch.

First time, try `EPOCHS = 1` and `FRACTION = 0.1` (it takes a few minutes) to make sure everything is saved to your Drive. Then set them back to `EPOCHS = 30` and `FRACTION = 1.0`. Quick test runs are saved in a separate folder (its name contains `TEST`), so they never mix with the real run.

## Who runs what

| Student | MODEL_TO_RUN | SEED |
|---|---|---|
| Maryam | YOLOv10s | 1 |
| Retal | YOLO11s | 1 |
| Joud Alsabaa | RT-DETR-L | 1 |
| Joud Fayez | RT-DETR-L | 0 |
| Lina | YOLO11s | 0 |
| Renad | YOLOv10s | 0 |

## What is saved in your Drive

Inside `MyDrive/result_of_fire_smoke_project/<model>_seed<N>/`:

- `train/run/`: `results.csv` (for the curves), `weights/best.pt`, `weights/last.pt`, and the training plots.
- `test_eval/`: the test-set results and the confusion matrix.
- `report.json` and `report.txt`: the metrics and the Optimization values.

## Notes

- Do not change any other setting. The protocol must be identical for everyone.
- Approximate time (from the smoke test): RT-DETR-L about 5.4 hours or more, YOLO11s about 1.3 hours, YOLOv10s about 1.5 hours.
- If the loss becomes NaN, Ultralytics reloads the last saved epoch (`last.pt`) and retries, up to 3 times. If it still fails, the notebook stops and prints a clear message. Do not simply re-run it; tell the team first.
- Dataset: downloaded automatically from our Google Drive link (DATASET_DRIVE_LINK). Figshare is only a backup because it often refuses downloads from Colab (403 Forbidden).

In [4]:
# ======================= SETTINGS =======================

# ---------- (1) CHANGE ONLY THESE LINES ----------
MODEL_TO_RUN = "YOLO11s"     # "RT-DETR-L"  |  "YOLO11s"  |  "YOLOv10s"
SEED         = 0              # 0 or 1
DATASET_DRIVE_LINK = "https://drive.google.com/file/d/1AzA0aGDKYlLA-Y-JcYNwIpm7MdEbixVW/view?usp=drive_link"
# -----------------------------------------------------

# ---------- (2) DO NOT CHANGE ANYTHING BELOW: the protocol must be identical for everyone ----------
EPOCHS    = 30
FRACTION  = 1.0               # share of the TRAINING images used (1.0 = all 8,826)
IMGSZ     = 640
WORKERS   = 2                 # free Colab has only 2 CPU cores
CACHE     = False             # "ram" will not fit 8.8k images in free Colab RAM
AMP       = True              # mixed precision (FP16 + FP32): faster, less GPU memory

# 'batches' = micro-batch sizes tried in order; the next one is used only after a CUDA out-of-memory.
# Same micro-batch (8) for every model. Ultralytics accumulates gradients over 8 micro-batches (nbs=64),
# so every model is updated after 64 images.
ALL_MODELS = {
    "RT-DETR-L": {"name": "RT-DETR-L", "weights": "rtdetr-l.pt", "cls": "RTDETR", "batches": [8, 4]},
    "YOLO11s":   {"name": "YOLO11s",   "weights": "yolo11s.pt",  "cls": "YOLO",   "batches": [8, 4]},
    "YOLOv10s":  {"name": "YOLOv10s",  "weights": "yolov10s.pt", "cls": "YOLO",   "batches": [8, 4]},
}
assert MODEL_TO_RUN in ALL_MODELS, f"MODEL_TO_RUN must be one of {list(ALL_MODELS)}, got {MODEL_TO_RUN!r}"
CFG = ALL_MODELS[MODEL_TO_RUN]

# --- dataset (from our Drive link; Figshare is only a backup)
DATASET_URL    = "https://ndownloader.figshare.com/files/53486522"   # official CQUniversity record, file MyData_Fire.zip
ZIP_PATH       = "/content/MyData_Fire.zip"
EXTRACT_DIR    = "/content/MyData_Fire"
EXPECTED_MD5   = "fd9ed42cd67fd5f7c016ca127985e245"                  # MD5 published on the Figshare record
EXPECTED_BYTES = 544994383
EXPECTED_SPLITS = {"train": 8826, "val": 1088, "test": 1107}         # sanity check only

# --- results go to MY Google Drive
RESULTS_DIR = "/content/drive/MyDrive/result_of_fire_smoke_project"

INSTALL_ULTRALYTICS = True
ULTRALYTICS_SPEC    = "ultralytics==8.4.173"
REQUIRE_GPU         = True

# --- name of this run's folder (a quick test never mixes with the real 30-epoch run)
import re
def _slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
RUN_NAME = f"{_slug(MODEL_TO_RUN)}_seed{SEED}"
if EPOCHS != 30 or FRACTION != 1.0:
    RUN_NAME += f"_TEST_{EPOCHS}ep_{int(FRACTION * 100)}pct"
print(f"Model: {MODEL_TO_RUN} | seed: {SEED} | epochs: {EPOCHS} | train fraction: {FRACTION} | results folder: {RUN_NAME}")

Model: YOLO11s | seed: 0 | epochs: 30 | train fraction: 1.0 | results folder: yolo11s_seed0


## Step 1 - Check the GPU

In [5]:
import os, sys, time, json, math, gc, shutil, subprocess, platform, traceback, datetime, random, threading
from pathlib import Path

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return ""

print("Python:", platform.python_version())
print(sh("nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv") or "nvidia-smi not available")

import torch
HAS_GPU = torch.cuda.is_available()
GPU_NAME, GPU_VRAM_MIB = None, None
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME, GPU_VRAM_MIB = props.name, props.total_memory / 2**20
    print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_MIB/1024:.1f} GiB | torch {torch.__version__} | CUDA {torch.version.cuda}")
    if "T4" not in GPU_NAME:
        print("WARNING: this is not a T4. Timings will NOT represent the free-tier T4 you plan to use.")
else:
    msg = "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then Runtime > Restart session and Run all."
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

import psutil
_disk_root = "/content" if os.path.exists("/content") else "/"
CPU_CORES = os.cpu_count()
RAM_GIB = psutil.virtual_memory().total / 2**30
print(f"CPU cores: {CPU_CORES} | RAM: {RAM_GIB:.1f} GiB | free disk: {shutil.disk_usage(_disk_root).free/2**30:.0f} GiB")

Python: 3.13.16
name, memory.total [MiB], memory.used [MiB], driver_version
Tesla T4, 15360 MiB, 0 MiB, 580.82.07
GPU: Tesla T4 | VRAM: 14.6 GiB | torch 2.14.1+cu130 | CUDA 13.0
CPU cores: 2 | RAM: 12.7 GiB | free disk: 113 GiB


## Step 2 - Install Ultralytics (pinned version)

In [6]:
if INSTALL_ULTRALYTICS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", ULTRALYTICS_SPEC], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError(f"pip install {ULTRALYTICS_SPEC} failed - see the messages above. "
                           "If that exact version cannot be installed, tell the team before changing ULTRALYTICS_SPEC "
                           "(a different version would make the timings not comparable with the earlier tests).")

try:
    from importlib.metadata import version as _pkg_version
    if _pkg_version("torch").split("+")[0] != torch.__version__.split("+")[0]:
        print("NOTE: pip changed the PyTorch version. Use Runtime > Restart session, then Run all again.")
except Exception:
    pass

os.environ.setdefault("WANDB_MODE", "disabled")
import ultralytics
from ultralytics import RTDETR, YOLO
from ultralytics.utils import LOGGER
ULTRALYTICS_VERSION = ultralytics.__version__
print("Ultralytics", ULTRALYTICS_VERSION, "| torch", torch.__version__)
_want = ULTRALYTICS_SPEC.split("==")[1] if "==" in ULTRALYTICS_SPEC else None
if _want and ULTRALYTICS_VERSION != _want:
    raise RuntimeError(f"Ultralytics {ULTRALYTICS_VERSION} is loaded but {_want} is required. "
                       "Use Runtime > Restart session, then Run all again.")
ultralytics.checks()

Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/112.6 GB disk)


## Step 3 - Connect Google Drive (required)


In [7]:
import re, logging
from google.colab import drive

try:
    drive.mount("/content/drive")
except Exception as e:
    print(f"Google Drive mount FAILED ({type(e).__name__}: {e}).")
    print("Fix: run this cell again and click 'Allow' on the Google popup (same Google account as Colab). "
          "If it still fails: Runtime > Disconnect and delete runtime, then Run all again.")
    raise

if not os.path.ismount("/content/drive"):
    print("WARNING: /content/drive does not look like a mounted Drive. Check that your Drive files appear in the Files panel before you continue.")

OUT_DIR = Path(RESULTS_DIR) / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / ".write_test").write_text("ok"); (OUT_DIR / ".write_test").unlink()
TRAIN_PROJECT = OUT_DIR / "train"


class LineCatcher(logging.Handler):
    # Keeps the few Ultralytics log lines we want to quote in the report: optimizer, AMP check, pretrained transfer, model size, resume.
    KEEP = re.compile(r"optimizer:|AMP:|Transferred \d+/\d+|Overriding model\.yaml|summary:|Resuming training", re.I)
    ANSI = re.compile(r"\x1b\[[0-9;]*m")
    def __init__(self):
        super().__init__()
        self.lines = []
    def emit(self, record):
        try:
            for part in self.ANSI.sub("", record.getMessage()).splitlines():
                if self.KEEP.search(part):
                    self.lines.append(part.strip())
        except Exception:
            pass

LOG_PATH = OUT_DIR / "ultralytics_train.log"
for _h in ("_fh", "CAP"):
    if _h in globals():
        LOGGER.removeHandler(globals()[_h])          # avoid duplicate handlers if this cell is run twice
_fh = logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8")   # append: a resumed run keeps the earlier log
_fh.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
LOGGER.addHandler(_fh)
CAP = LineCatcher()
LOGGER.addHandler(CAP)
print("Results will be saved to (your Google Drive):", OUT_DIR)

Mounted at /content/drive
Results will be saved to (your Google Drive): /content/drive/MyDrive/result_of_fire_smoke_project/yolo11s_seed0


## Step 4 - Get the dataset

The dataset (`MyData_Fire.zip`) is taken from the first place that works:

1. **Your Google Drive**, if the file is already there (any folder).
2. **Our shared Drive link** (`DATASET_DRIVE_LINK`).
3. **Figshare** (the official source), only as a backup. It often refuses downloads from Colab (`403 Forbidden`).

In [8]:
# Finds MyData_Fire.zip in your Drive (or downloads it from DATASET_DRIVE_LINK) and copies it to Colab.
# Step 5 then verifies the MD5, so a wrong or broken file can never be used.
_zip = Path(ZIP_PATH)
MY_DRIVE = Path(RESULTS_DIR).parent            # /content/drive/MyDrive

def _right_size(p):
    try:
        return p.stat().st_size == EXPECTED_BYTES
    except OSError:
        return False

def _find_in_drive(root, max_depth=3):
    for cur, dirs, files in os.walk(root):
        depth = len(Path(cur).relative_to(root).parts)
        dirs[:] = [] if depth >= max_depth else [d for d in dirs if not d.startswith(".") and   d != Path(RESULTS_DIR).name]
        for f in files:
            if "mydata_fire" in f.lower() and f.lower().endswith(".zip") and _right_size(Path(cur) / f):
                return Path(cur) / f
    return None

if _zip.exists() and _right_size(_zip):
    print("The dataset zip is already on Colab:", _zip)
else:
    if _zip.exists():
        print(f"{_zip} is incomplete ({_zip.stat().st_size} bytes, expected {EXPECTED_BYTES}) -> removing it")
        _zip.unlink()
    found = _find_in_drive(MY_DRIVE) if MY_DRIVE.is_dir() else None
    if found:
        print(f"Found the dataset in your Drive: {found}\nCopying to {_zip} (about 1 minute) ...")
        _t0 = time.perf_counter()
        shutil.copy(found, _zip)
        print(f"Copied in {time.perf_counter() - _t0:.0f} s")
    elif DATASET_DRIVE_LINK.strip():
        print("Not found in your Drive -> downloading from DATASET_DRIVE_LINK ...")
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
            import gdown
        try:
            gdown.download(url=DATASET_DRIVE_LINK.strip(), output=str(_zip), quiet=False, fuzzy=True)
        except Exception as e:
            print(f"Download from the Drive link FAILED ({type(e).__name__}: {e}). "
                  "Check that the link is shared as 'Anyone with the link'.")
    else:
        top = sorted(p.name for p in MY_DRIVE.iterdir())[:30] if MY_DRIVE.is_dir() else []
        print("MyData_Fire.zip was NOT found in the Google Drive that Colab is connected to.")
        if any("mydata_fire" in n.lower() for n in top):
            print(f"A file with that name IS there, but its size is not {EXPECTED_BYTES} bytes: the upload is incomplete. Upload it again.")
        print("That Drive contains (top folder):", top)
        print("If your file is not in this list, you chose a DIFFERENT Google account in the Drive popup:")
        print("  Runtime > Disconnect and delete runtime, then Run all, and choose the account that has the file.")
        print("Step 5 will now try Figshare (it may fail with 403).")

Not found in your Drive -> downloading from DATASET_DRIVE_LINK ...


Downloading...
From (original): https://drive.google.com/uc?id=1AzA0aGDKYlLA-Y-JcYNwIpm7MdEbixVW
From (redirected): https://drive.google.com/uc?id=1AzA0aGDKYlLA-Y-JcYNwIpm7MdEbixVW&confirm=t&uuid=6f0d26af-c976-48a5-9634-c6623d9661b8
To: /content/MyData_Fire.zip
100%|██████████| 545M/545M [00:05<00:00, 93.4MB/s]


## Step 5 - Verify, extract and prepare the dataset

In [9]:
import zipfile, yaml, hashlib, collections, requests
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ZIP_P, EXTRACT_P = Path(ZIP_PATH), Path(EXTRACT_DIR)
WORK = ZIP_P.parent / "feasibility_work"
RUNS_DIR = ZIP_P.parent / "runs_feasibility"
WORK.mkdir(parents=True, exist_ok=True)
DOWNLOAD_INFO = {"url": DATASET_URL, "expected_md5": EXPECTED_MD5}

def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def download_verified(url, dest, expected_md5, expected_bytes=None, attempts=3):
    dest = Path(dest)
    if dest.exists():
        print(f"{dest} already exists - verifying MD5 ...")
        got = md5_of(dest)
        if got == expected_md5:
            print("MD5 OK (existing file):", got)
            DOWNLOAD_INFO.update(reused_existing_file=True, md5=got, md5_ok=True, bytes=dest.stat().st_size)
            return
        print(f"Existing file has MD5 {got}, expected {expected_md5} -> deleting it and downloading again")
        dest.unlink()
    part = dest.with_name(dest.name + ".part")
    last_err = None
    for k in range(1, attempts + 1):
        try:
            print(f"Download attempt {k}/{attempts}: {url}")
            t0 = time.perf_counter(); h = hashlib.md5(); n = 0; next_report = 0
            with requests.get(url, stream=True, timeout=(20, 60), headers={"User-Agent": "Mozilla/5.0"}) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0)) or expected_bytes
                with open(part, "wb") as out:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        if not chunk:
                            continue
                        out.write(chunk); h.update(chunk); n += len(chunk)
                        if total and n >= next_report:
                            print(f"  {n/2**20:6.0f} / {total/2**20:.0f} MiB ({100*n/total:3.0f} %)  {n/2**20/max(time.perf_counter()-t0, 1e-6):5.1f} MiB/s")
                            next_report += total * 0.1
            secs = time.perf_counter() - t0
            md5 = h.hexdigest()
            if expected_bytes and n != expected_bytes:
                raise IOError(f"received {n} bytes, expected {expected_bytes}")
            if md5 != expected_md5:
                raise ValueError(f"MD5 mismatch: got {md5}, expected {expected_md5}")
            part.replace(dest)
            print(f"Downloaded {n/2**20:.0f} MiB in {secs:.0f} s. MD5 OK: {md5}")
            DOWNLOAD_INFO.update(reused_existing_file=False, md5=md5, md5_ok=True, bytes=n, seconds=round(secs, 1))
            return
        except Exception as e:
            last_err = e
            print(f"  attempt {k} failed: {type(e).__name__}: {str(e)[:200]}")
            part.unlink(missing_ok=True)
            time.sleep(5)
    raise RuntimeError(f"Could not download a valid copy after {attempts} attempts (last error: {last_err}).\n"
                       f"Open {url} in your browser, upload the file to {dest} using the Files panel, and re-run this cell.")

download_verified(DATASET_URL, ZIP_P, EXPECTED_MD5, EXPECTED_BYTES)

# ---- extract to /content/MyData_Fire (skipped if a complete extraction is already there)
marker = EXTRACT_P / ".extract_complete"
if marker.exists():
    print("Already extracted:", EXTRACT_P)
else:
    if EXTRACT_P.exists():
        shutil.rmtree(EXTRACT_P)                       # an earlier extraction was interrupted
    t0 = time.perf_counter(); print(f"Extracting to {EXTRACT_P} ...")
    with zipfile.ZipFile(ZIP_P) as z:
        z.extractall(EXTRACT_P)
    marker.write_text("ok")
    print(f"Extracted in {time.perf_counter()-t0:.0f} s")

# ---- locate data.yaml, keep the original, rewrite it with working paths
yamls = sorted(EXTRACT_P.rglob("data.yaml"), key=lambda p: len(p.parts))
ds_root = yamls[0].parent if yamls else EXTRACT_P
orig_yaml = ds_root / "data.original.yaml"
if yamls and not orig_yaml.exists():
    shutil.copy(yamls[0], orig_yaml)
orig_cfg = yaml.safe_load(open(orig_yaml)) if orig_yaml.exists() else {}
if not yamls:
    print("WARNING: no data.yaml found in the archive; building one from the folder layout.")
print("\nDataset root:", ds_root)
print("BEFORE (original data.yaml):", {k: orig_cfg[k] for k in ("train", "val", "test", "nc", "names") if k in orig_cfg})

SPLIT_NAMES = {"train": ["train"], "val": ["valid", "val", "validation"], "test": ["test"]}
split_dirs = {}
for key, names_ in SPLIT_NAMES.items():
    found = None
    for n_ in names_:
        if (ds_root / n_ / "images").is_dir():
            found = ds_root / n_ / "images"; break
    if found is None:
        for p in ds_root.rglob("images"):
            if p.is_dir() and p.parent.name.lower() in names_:
                found = p; break
    split_dirs[key] = found
if split_dirs["train"] is None or split_dirs["val"] is None:
    raise RuntimeError(f"Could not find train/ and valid/ image folders under {ds_root}. Found: {split_dirs}")

names = orig_cfg.get("names", ["fire", "smoke"])
_n = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
if _n != ["fire", "smoke"]:
    print("WARNING: class names are", names, "- expected fire, smoke.")

fixed = dict(orig_cfg)                                  # keeps nc, names and the roboflow block
fixed.update({"path": str(ds_root), "train": str(split_dirs["train"]), "val": str(split_dirs["val"]), "names": names, "nc": len(_n)})
if split_dirs["test"] is not None:
    fixed["test"] = str(split_dirs["test"])
else:
    fixed.pop("test", None)
yaml.safe_dump(fixed, open(ds_root / "data.yaml", "w"), sort_keys=False)
print("AFTER  (fixed data.yaml)   :", {k: fixed[k] for k in ("path", "train", "val", "test", "nc", "names") if k in fixed})

def list_images(d):
    return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXT)

DATASET_STATS = {}
print("\nsplit  images  labels   class0   class1   expected  ok?")
for key, d in split_dirs.items():
    if d is None:
        continue
    imgs = list_images(d)
    lbl_dir = d.parent / "labels"
    lbl_files = sorted(lbl_dir.glob("*.txt")) if lbl_dir.is_dir() else []
    inst = collections.Counter()
    for lf in lbl_files:
        for line in lf.read_text().splitlines():
            if line.strip():
                inst[line.split()[0]] += 1
    DATASET_STATS[key] = {"images": len(imgs), "labels": len(lbl_files), "instances": {k_: inst[k_] for k_ in sorted(inst)}}
    exp = EXPECTED_SPLITS.get(key)
    ok = "yes" if exp == len(imgs) else f"NO (expected {exp})"
    print(f"{key:<6} {len(imgs):>6} {len(lbl_files):>7} {inst['0']:>8} {inst['1']:>8} {exp!s:>9}  {ok}")

train_imgs = list_images(split_dirs["train"])
sample_sizes = collections.Counter(Image.open(p).size for p in random.Random(0).sample(train_imgs, min(200, len(train_imgs))))
print("Image sizes in a 200-image sample:", dict(sample_sizes))

# ---- feasibility YAML: random subset of the training images, complete validation split
k = max(1, round(FRACTION * len(train_imgs)))
subset = sorted(random.Random(SEED).sample(train_imgs, k))
subset_txt = WORK / "train_subset.txt"
subset_txt.write_text("\n".join(str(p.resolve()) for p in subset) + "\n")
cfg = {"path": str(ds_root), "train": str(subset_txt), "val": str(split_dirs["val"]), "names": names, "nc": len(_n)}
if split_dirs["test"] is not None:
    cfg["test"] = str(split_dirs["test"])
YAML_PATH = WORK / "fire_feasibility.yaml"
yaml.safe_dump(cfg, open(YAML_PATH, "w"), sort_keys=False)
for src_file in (YAML_PATH, subset_txt, ds_root / "data.yaml", orig_yaml):
    if src_file.exists():
        shutil.copy(src_file, OUT_DIR / (src_file.name if src_file.name != "data.yaml" else "data_fixed.yaml"))

print(f"\nTraining subset: {k} of {len(train_imgs)} training images ({100*k/len(train_imgs):.1f} %), validation: {DATASET_STATS['val']['images']} images")
print("Feasibility YAML:", YAML_PATH); print(open(YAML_PATH).read())

/content/MyData_Fire.zip already exists - verifying MD5 ...
MD5 OK (existing file): fd9ed42cd67fd5f7c016ca127985e245
Extracting to /content/MyData_Fire ...
Extracted in 7 s

Dataset root: /content/MyData_Fire
BEFORE (original data.yaml): {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 2, 'names': ['fire', 'smoke']}
AFTER  (fixed data.yaml)   : {'path': '/content/MyData_Fire', 'train': '/content/MyData_Fire/train/images', 'val': '/content/MyData_Fire/valid/images', 'test': '/content/MyData_Fire/test/images', 'nc': 2, 'names': ['fire', 'smoke']}

split  images  labels   class0   class1   expected  ok?
train    8826    8826    10090     9724      8826  yes
val      1088    1088     1241     1255      1088  yes
test     1107    1107     1258     1247      1107  yes
Image sizes in a 200-image sample: {(640, 640): 200}

Training subset: 8826 of 8826 training images (100.0 %), validation: 1088 images
Feasibility YAML: /content/feasibility_work/fire_feasi

## Step 6 - Train (30 epochs)

- The checkpoint (`last.pt`) is saved to your Drive every epoch.
- If Colab disconnects, run the notebook again from the top (Run all). It resumes from the last saved epoch.
- If the loss becomes NaN/inf, Ultralytics reloads `last.pt` and retries the epoch (up to 3 times). The notebook only records it. If Ultralytics gives up, the notebook stops.

In [10]:
import csv

MODEL_CLASSES = {"RTDETR": RTDETR, "YOLO": YOLO}
RUN_DIR   = TRAIN_PROJECT / "run"
LAST_PT   = RUN_DIR / "weights" / "last.pt"
BEST_PT   = RUN_DIR / "weights" / "best.pt"
FAIL_MARK = OUT_DIR / "FAILED_NAN.txt"
NONFINITE = {"count": 0, "epochs": []}

if FAIL_MARK.exists():
    raise RuntimeError("A previous run in this folder was stopped because of NaN/inf losses (see FAILED_NAN.txt). "
                       "Tell the team before trying again. To start over, delete FAILED_NAN.txt and the 'train' folder in your results folder.")

def is_oom(exc):
    return isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower()

def free_gpu():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()
    time.sleep(2)

def loss_values(trainer):
    try:
        li = getattr(trainer, "loss_items", None)
        if li is None:
            return []
        if isinstance(li, dict):
            vals = list(li.values())
        elif hasattr(li, "detach"):
            vals = li.detach().float().cpu().flatten().tolist()
        else:
            vals = list(li)
        return [float(v) for v in vals]
    except Exception:
        return []

def cb_batch_end(trainer):            # only OBSERVES and records NaN/inf losses; Ultralytics itself reloads last.pt and retries the epoch
    try:
        vals = loss_values(trainer)
        if vals and not all(math.isfinite(v) for v in vals):
            ep = int(getattr(trainer, "epoch", 0)) + 1
            NONFINITE["count"] += 1
            if ep not in NONFINITE["epochs"]:
                NONFINITE["epochs"].append(ep)
                print(f"\n!! WARNING: NaN/inf loss in epoch {ep}. Ultralytics will reload last.pt and retry this epoch (up to 3 times).")
    except Exception:
        pass

def is_nan_failure(exc):           # Ultralytics gave up after its NaN recovery attempts
    return bool(re.search(r"\bnan\b|non-finite", str(exc), re.I)) or NONFINITE["count"] > 0

def mark_failed(exc):
    FAIL_MARK.write_text(f"Stopped: Ultralytics could not recover from NaN/inf losses (epochs: {NONFINITE['epochs']}).\n{type(exc).__name__}: {exc}\n")
    raise RuntimeError("Training was STOPPED: the loss stayed NaN/inf after Ultralytics' 3 recovery attempts. Do NOT just re-run. "
                       f"Tell the team (epochs: {NONFINITE['epochs']}) and send the file {LOG_PATH.name}.") from exc

def make_model(weights):
    m = MODEL_CLASSES[CFG["cls"]](weights)
    m.add_callback("on_train_batch_end", cb_batch_end)
    return m

def checkpoint_state(path):
    """'none' = no usable checkpoint, 'unfinished' = can be resumed, 'finished' = training already completed."""
    if not Path(path).exists():
        return "none"
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        return "finished" if ck.get("epoch", -1) == -1 else "unfinished"
    except Exception as e:
        print(f"WARNING: cannot read {path} ({type(e).__name__}); this run will start from scratch.")
        return "none"

state = checkpoint_state(LAST_PT)
print("Checkpoint state in Drive:", state)
t_start = time.perf_counter()
BATCH_USED = None

if state == "finished":
    print("Training of this run was already completed earlier -> skipping training and going to the evaluation.")

elif state == "unfinished":
    print("Found an unfinished run -> RESUMING from", LAST_PT)
    model = make_model(str(LAST_PT))
    try:
        model.train(resume=True)
    except Exception as e:
        if is_nan_failure(e):
            mark_failed(e)
        raise
    model = None
    free_gpu()

else:
    for b in CFG["batches"]:
        print(f"\n==================== {CFG['name']} | seed {SEED} | micro-batch {b} ====================")
        oom = False
        try:
            model = make_model(CFG["weights"])
            # identical protocol for every model: only the weights file and the micro-batch size differ
            model.train(data=str(YAML_PATH), epochs=EPOCHS, imgsz=IMGSZ, batch=b, workers=WORKERS,
                        device=0 if HAS_GPU else "cpu", amp=AMP, seed=SEED, deterministic=False,
                        cache=CACHE, plots=True, val=True,
                        project=str(TRAIN_PROJECT), name="run", exist_ok=True)
            BATCH_USED = b
        except Exception as e:
            if is_oom(e) and b != CFG["batches"][-1]:
                oom = True
                print(f"CUDA out of memory with batch {b} -> trying the next batch size ...")
            elif is_nan_failure(e):
                mark_failed(e)
            else:
                raise
        model = None
        free_gpu()                      # the exception (and its GPU tensors) is released here
        if not oom:
            break

TRAIN_MIN = (time.perf_counter() - t_start) / 60
print(f"\nTraining step finished in {TRAIN_MIN:.1f} min (this session only).")

if not BEST_PT.exists():
    raise RuntimeError(f"best.pt was not found at {BEST_PT}. Check the training log above.")
if NONFINITE["count"]:
    print(f"NOTE: NaN/inf loss in epoch(s) {NONFINITE['epochs']} ({NONFINITE['count']} batches). Ultralytics recovered from last.pt "
          "and training finished. Mention this to the team (it goes in the Methodology).")

Checkpoint state in Drive: none

==================== YOLO11s | seed 0 | micro-batch 8 ====================
New https://pypi.org/project/ultralytics/8.4.209 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/feasibility_work/fire_feasibility.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.

## Step 7 - Evaluate the best checkpoint on the TEST split and save the report


`best.pt` is the epoch with the best validation score. It is evaluated once on the test split (1,107 images).

In [11]:
# ---------------- evaluation on the test split ----------------
EVAL_MODEL = MODEL_CLASSES[CFG["cls"]](str(BEST_PT))
metrics = EVAL_MODEL.val(data=str(YAML_PATH), split="test", imgsz=IMGSZ, batch=8, workers=WORKERS,
                         device=0 if HAS_GPU else "cpu", plots=True,
                         project=str(OUT_DIR), name="test_eval", exist_ok=True)

TEST = {}
try:
    TEST["precision"] = float(metrics.box.mp)
    TEST["recall"]    = float(metrics.box.mr)
    TEST["mAP50"]     = float(metrics.box.map50)
    TEST["mAP50_95"]  = float(metrics.box.map)
    names = metrics.names
    TEST["per_class"] = {}
    for j, ci in enumerate(metrics.box.ap_class_index):
        p, r, ap50, ap = metrics.box.class_result(j)
        TEST["per_class"][str(names[int(ci)])] = {"precision": float(p), "recall": float(r), "AP50": float(ap50), "AP50_95": float(ap)}
    TEST["speed_ms_per_image"] = {k: round(float(v), 2) for k, v in metrics.speed.items()}
except Exception as e:
    TEST["error"] = f"{type(e).__name__}: {e}"
    print("WARNING: could not read all the test metrics:", TEST["error"])

# ---------------- model size ----------------
MODEL_INFO = {"best_pt_MB": round(BEST_PT.stat().st_size / 2**20, 1)}
try:
    MODEL_INFO["parameters_M"] = round(sum(p.numel() for p in EVAL_MODEL.model.parameters()) / 1e6, 2)
except Exception:
    pass

# ---------------- Optimization values (for Section 4 of the Methodology) ----------------
KEEP_ARGS = ("optimizer", "lr0", "lrf", "momentum", "weight_decay", "nbs", "batch", "epochs", "imgsz", "amp", "seed",
             "deterministic", "warmup_epochs", "close_mosaic", "patience", "cos_lr")
def ckpt_args(path):
    try:
        a = (torch.load(path, map_location="cpu", weights_only=False).get("train_args") or {})
        return {k: a.get(k) for k in KEEP_ARGS}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}
TRAIN_ARGS = ckpt_args(LAST_PT)
OPT_LINE = next((l for l in reversed(CAP.lines) if "optimizer:" in l.lower() and "(lr=" in l), None)   # the line with the optimizer actually used

# ---------------- validation curve summary (from results.csv) ----------------
VAL = {}
try:
    with open(RUN_DIR / "results.csv", newline="") as f:
        rows = [{(k or "").strip(): (v or "").strip() for k, v in r.items()} for r in csv.DictReader(f)]
    VAL["epochs_in_results_csv"] = len(rows)
    fit = lambda r: 0.1 * float(r["metrics/mAP50(B)"]) + 0.9 * float(r["metrics/mAP50-95(B)"])
    pick = lambda r: {"epoch": int(float(r["epoch"])), "mAP50": float(r["metrics/mAP50(B)"]), "mAP50_95": float(r["metrics/mAP50-95(B)"])}
    VAL["last_epoch"] = pick(rows[-1])
    VAL["best_epoch"] = pick(max(rows, key=fit))
except Exception as e:
    VAL["error"] = f"{type(e).__name__}: {e}"

REPORT = {
    "run_name": RUN_NAME, "model": MODEL_TO_RUN, "seed": SEED, "epochs": EPOCHS, "fraction_of_train": FRACTION,
    "micro_batch_used_in_this_session": BATCH_USED,
    "environment": {"gpu": GPU_NAME, "ultralytics": ULTRALYTICS_VERSION, "torch": torch.__version__},
    "dataset": DATASET_STATS, "dataset_md5_ok": DOWNLOAD_INFO.get("md5_ok"),
    "train_args": TRAIN_ARGS, "optimizer_log_line": OPT_LINE,
    "nonfinite_training_batches": NONFINITE["count"], "nan_epochs_recovered": NONFINITE["epochs"],
    "training_minutes_this_session": round(TRAIN_MIN, 1),
    "validation": VAL, "test": TEST, "model_info": MODEL_INFO,
}
(OUT_DIR / "report.json").write_text(json.dumps(REPORT, indent=2, default=str))

L = ["=" * 90, f"FINAL RUN REPORT - {MODEL_TO_RUN} - seed {SEED}", "=" * 90,
     f"GPU: {GPU_NAME} | Ultralytics {ULTRALYTICS_VERSION} | torch {torch.__version__}",
     f"Epochs: {EPOCHS} | train fraction: {FRACTION} | NaN/inf batches: {NONFINITE['count']} (epochs {NONFINITE['epochs']})",
     f"Optimizer line : {OPT_LINE}",
     f"Train args     : {TRAIN_ARGS}",
     f"Validation     : {VAL}",
     f"TEST (best.pt) : precision {TEST.get('precision', float('nan')):.4f} | recall {TEST.get('recall', float('nan')):.4f} | "
     f"mAP50 {TEST.get('mAP50', float('nan')):.4f} | mAP50-95 {TEST.get('mAP50_95', float('nan')):.4f}"]
for cname, v in (TEST.get("per_class") or {}).items():
    L.append(f"   {cname:<6}: P {v['precision']:.4f} | R {v['recall']:.4f} | AP50 {v['AP50']:.4f} | AP50-95 {v['AP50_95']:.4f}")
L += [f"Speed (ms/image, T4): {TEST.get('speed_ms_per_image')}",
      f"Model: {MODEL_INFO}", "=" * 90]
SUMMARY_TEXT = "\n".join(L)
(OUT_DIR / "report.txt").write_text(SUMMARY_TEXT)
print(SUMMARY_TEXT)
print("\nSaved to your Drive:", OUT_DIR)
print("Folder content:", sorted(p.name for p in OUT_DIR.iterdir()))

Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 123.6±242.9 MB/s, size: 75.0 KB)
val: Scanning /content/MyData_Fire/test/labels... 1107 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1107/1107 761.8it/s 1.5s
val: New cache created: /content/MyData_Fire/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 139/139 7.7it/s 17.9s
                   all       1107       2505      0.647      0.514      0.584      0.319
                  fire        692       1258       0.64      0.553      0.631      0.346
                 smoke        712       1247      0.655      0.474      0.537      0.292
Speed: 1.4ms preprocess, 9.4ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/drive/MyDrive/result_of_fire_smoke_project/yolo11

## What to do next

1. Copy the summary printed above (or the file `report.txt`) and send it to the team.
2. Share the `result_of_fire_smoke_project` folder from your Drive with the team (Share, then add their emails).
3. Retal needs `train/run/results.csv` and `train/run/weights/best.pt` from every run, for the curves and the metrics.